# Этап 03. Проверить полученные данные на исходной базе
Покрытие — относительно всех 2016 МО, отдельно по 2023 и 2024. Пропуски не становятся нулями.
Выбор исходного показателя с высоким покрытием ещё не гарантирует пригодность отношения: четвёртый ноутбук заново проверит покрытие после соединения числителя и знаменателя.


In [1]:
from pathlib import Path
import json, sys
import pandas as pd
from IPython.display import display
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.bdmo_workflow import IndicatorStore, base_panel, load_mapping
from src.recommendations import read_plan, recommended_queue, search_catalog
config = json.loads((ROOT / 'configs/bdmo.json').read_text(encoding='utf-8'))
OUTPUT_DIR = ROOT / config['output_dir']
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
panel, base_ids = base_panel(ROOT / config['base_panel'], config['expected_municipalities'], **config['calendar'])
mapping = load_mapping(ROOT / config['mapping_path'], base_ids, config['history_years'])
store = IndicatorStore(ROOT, config)
plan = read_plan(ROOT, config)
pd.set_option('display.max_colwidth', 120)
print(f'База: {len(base_ids)} МО x 24 месяца')


База: 2016 МО x 24 месяца


In [2]:
from src.audit import indicator_coverage, assign_status, choose_indicators, basic_checks
from src.bdmo_workflow import coverage_against_base
data = pd.read_parquet(OUTPUT_DIR / 'bdmo_long.parquet')
candidates = pd.read_csv(OUTPUT_DIR / 'indicator_candidates.csv', keep_default_na=False, dtype={'indicator_code':str})
display(basic_checks(data, config['focus_years']))
coverage = indicator_coverage(data, candidates, config['focus_years'], base_mapping=mapping)
audit = assign_status(coverage, config['core_min_coverage'], config['sensitivity_min_coverage'])
display(audit)
audit.to_csv(OUTPUT_DIR / 'indicator_audit.csv', index=False)
selected = choose_indicators(audit, config.get('indicator_overrides'))
selected.to_csv(OUTPUT_DIR / 'selected_indicators.csv', index=False)


,check,value
0,rows,43308
1,municipalities,2094
2,years,"2023, 2024"
3,focus_years_present,True
4,missing_oktmo,0
5,duplicate_keys,0
6,missing_values,0


,feature_key,view,requested_indicator,match_score,units,n_units,unit_missing,2023,2024,min_focus_coverage,status
0,population_mean,demography,Среднегодовая численность постоянного населения,0,[Человек],1,False,0.958829,0.917659,0.917659,core
1,population_jan,demography,Оценка численности населения на 1 января текущего года,0,[Человек],1,False,0.997520,0.999504,0.997520,core
2,land_area_ha,demography,Общая площадь земель муниципального образования,0,[Гектар],1,False,1.000000,0.990575,0.990575,core
3,wage,labor,Среднемесячная заработная плата работников организаций (без субъектов малого предпринимательства),0,[Рубль],1,False,1.000000,0.992560,0.992560,core
4,employment,labor,Среднесписочная численность работников организаций (без субъектов малого предпринимательства),0,[Человек],1,False,1.000000,0.992560,0.992560,core
5,reporting_companies,business,Количество хозяйствующих субъектов по данным бухгалтерской отчетности,0,[Единица],1,False,1.000000,0.996032,0.996032,core
6,profitable_share,business,Удельный вес прибыльных организаций в общем числе организаций по данным бухгалтерской отчетности,0,[Процент],1,False,1.000000,0.992063,0.992063,core
7,investment_total,investment,"Инвестиции в основной капитал, осуществляемые организациями, находящимися на территории муниципального образования (...",0,[Тысяча рублей],1,False,0.996528,0.992063,0.992063,core
8,shop_floor,retail_supply,Площадь торгового зала объектов розничной торговли,0,[Квадратный метр],1,False,0.933532,0.936012,0.933532,core
9,lit_street_km,infrastructure,"Общая протяженность освещенных частей улиц, проездов, набережных",0,"[Километр, тысяча метров]",1,False,0.949405,0.939980,0.939980,core


In [3]:
reports, missing_parts = [], []
for meta in candidates.to_dict('records'):
    local = data[data.feature_key.eq(meta['feature_key']) & data.view.eq(meta['view']) & data.requested_indicator.eq(meta['indicator_name'])]
    report, missing = coverage_against_base(local, mapping, meta, config['focus_years'])
    reports.append(report); missing_parts.append(missing)
pd.concat(reports).to_csv(OUTPUT_DIR / 'coverage_by_year.csv', index=False)
pd.concat(missing_parts).to_parquet(OUTPUT_DIR / 'missing_base.parquet', index=False)
display(pd.read_parquet(OUTPUT_DIR / 'conflicts.parquet').head(20))


,oktmo,year,feature_key,view,requested_indicator,reason
0,46744000,2023,population_mean,demography,Среднегодовая численность постоянного населения,conflicting_values_or_units
1,46755000,2023,population_mean,demography,Среднегодовая численность постоянного населения,conflicting_values_or_units
2,54501000,2023,population_mean,demography,Среднегодовая численность постоянного населения,conflicting_values_or_units
3,54501000,2024,population_mean,demography,Среднегодовая численность постоянного населения,conflicting_values_or_units
4,54604000,2023,population_mean,demography,Среднегодовая численность постоянного населения,conflicting_values_or_units
5,54604000,2024,population_mean,demography,Среднегодовая численность постоянного населения,conflicting_values_or_units
6,54608000,2023,population_mean,demography,Среднегодовая численность постоянного населения,conflicting_values_or_units
7,54608000,2024,population_mean,demography,Среднегодовая численность постоянного населения,conflicting_values_or_units
8,54610000,2023,population_mean,demography,Среднегодовая численность постоянного населения,conflicting_values_or_units
9,54610000,2024,population_mean,demography,Среднегодовая численность постоянного населения,conflicting_values_or_units
